In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import joblib as jb

In [5]:
df = pd.read_csv("telecom_churn.csv")

In [9]:
df.head()

,Churn,AccountWeeks,ContractRenewal,DataPlan,DataUsage,CustServCalls,DayMins,DayCalls,MonthlyCharge,OverageFee,RoamMins
0,0,128,1,1,2.7,1,265.1,110,89.0,9.87,10.0
1,0,107,1,1,3.7,1,161.6,123,82.0,9.78,13.7
2,0,137,1,0,0.0,0,243.4,114,52.0,6.06,12.2
3,0,84,0,0,0.0,2,299.4,71,57.0,3.10,6.6
4,0,75,0,0,0.0,3,166.7,113,41.0,7.42,10.1


In [10]:
x = df.drop(['Churn'], axis=1)
y = df['Churn']

In [11]:
numerical_cols = x.select_dtypes(include=['int64','float64']).columns.tolist()

In [12]:
categorical_cols = x.select_dtypes(include=['object']).columns.tolist()

In [13]:
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

In [14]:
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

In [15]:
preprocessor = ColumnTransformer(transformers=[
    ('num',numerical_transformer,numerical_cols ),
    ('cate',categorical_transformer,categorical_cols )
])

In [16]:
X_train,X_test,y_train,y_test = train_test_split(x,y,test_size = 0.2,random_state = 42)

In [18]:
model = Pipeline(steps=[
    ('pre',preprocessor ),('log',LogisticRegression(max_iter=1000))
])

In [19]:
model.fit(X_train,y_train)

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['AccountWeeks',
                                                   'ContractRenewal',
                                                   'DataPlan', 'DataUsage',
                                                   'CustServCalls', 'DayMins',
                                                   'DayCalls', 'MonthlyCharge',
                                                   'OverageFee', 'RoamMins']),
                                                 ('cate',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  [])])),
                ('log', LogisticRegression(max_iter=1000))])

In [21]:
y_pred = model.predict(X_test)
print(f'{classification_report(y_test,y_pred,zero_division = 0)}')

              precision    recall  f1-score   support

           0       0.87      0.98      0.92       566
           1       0.62      0.18      0.28       101

    accuracy                           0.86       667
   macro avg       0.75      0.58      0.60       667
weighted avg       0.83      0.86      0.82       667



In [26]:
jb.dump(model,'LogisticRegression.pkl')

['LogisticRegression.pkl']